# RQ3 Syntactic Entity Changes

Create the grouped entity table for paper Table IV.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "datasets/processed/rq3/workflow_syntactic_entities.csv").exists():
    ROOT = ROOT.parent

DATA = ROOT / "datasets/processed/rq3/workflow_syntactic_entities.csv"
OUT_DIR = ROOT / "datasets/processed/rq3"

df = pd.read_csv(DATA)
print(df.shape)


(1692, 11)


## Grouping

Map normalized GAWD paths to the Table IV entities. Whole steps and their standard keys are grouped as `jobs.<job_id>.steps[<index>]`. APR and HPR percentages use their respective PR-type totals; the pooled column is retained for package validation.

In [2]:
WORKFLOW_LEVEL_KEYS = {"name", "env", "concurrency", "on", "permissions"}
JOB_LEVEL_KEYS = {
    "name", "needs", "env", "with", "container", "timeout-minutes",
    "services", "defaults", "runs-on", "permissions", "if", "strategy",
    "outputs",
}
STEP_LEVEL_KEYS = {
    "name", "id", "if", "uses", "run", "shell", "with", "env",
    "continue-on-error", "timeout-minutes", "working-directory",
}
STATE_LABEL = {"added": "Added", "modified": "Changed", "removed": "Removed"}


def entity_row_key(keyword):
    if keyword == "jobs.<job_id>":
        return keyword
    if keyword.startswith("jobs.<job_id>.steps["):
        if "]." not in keyword:
            return "jobs.<job_id>.steps[<index>]"
        key = keyword.split("].", 1)[1].split(".")[0].split("[")[0]
        if key in STEP_LEVEL_KEYS:
            return "jobs.<job_id>.steps[<index>]"
        return "others"
    if keyword.startswith("jobs.<job_id>."):
        key = keyword[len("jobs.<job_id>."):].split(".")[0].split("[")[0]
        return f"jobs.<job_id>.{key}" if key in JOB_LEVEL_KEYS else "others"
    base = keyword.split(".")[0].split("[")[0]
    return base if base in WORKFLOW_LEVEL_KEYS else "others"


entity_df = df.copy()
entity_df["entity"] = entity_df["keyword"].map(entity_row_key)

counts = entity_df.pivot_table(
    index="entity",
    columns=["pr_type", "keyword_change_state"],
    aggfunc="size",
    fill_value=0,
)

total_changes = len(entity_df)
group_totals = {
    "ai": entity_df["pr_type"].eq("ai").sum(),
    "human": entity_df["pr_type"].eq("human").sum(),
}
pooled_pct = counts / total_changes * 100
group_divisor = pd.Series(
    [group_totals[pr_type] for pr_type, _ in counts.columns],
    index=counts.columns,
)
within_group_pct = counts.div(group_divisor, axis=1) * 100

entity_table = pd.DataFrame(index=within_group_pct.index)
for pr_type, column_prefix in [("ai", "APR"), ("human", "HPR")]:
    raw_columns = []
    for state in ["added", "modified", "removed"]:
        column = within_group_pct.get(
            (pr_type, state), pd.Series(0.0, index=within_group_pct.index)
        )
        raw_columns.append(column)
        entity_table[f"{column_prefix} {STATE_LABEL[state]}"] = column.round(1)
    entity_table[f"{column_prefix} Total"] = sum(raw_columns).round(1)

entity_table["All PRs Total (pooled)"] = sum(
    pooled_pct.get((pr_type, state), pd.Series(0.0, index=within_group_pct.index))
    for pr_type in ["ai", "human"]
    for state in ["added", "modified", "removed"]
).round(1)

entity_order = (
    ["name", "env", "concurrency", "on", "permissions", "jobs.<job_id>"]
    + [f"jobs.<job_id>.{key}" for key in [
        "name", "needs", "env", "with", "container", "timeout-minutes",
        "services", "defaults", "runs-on", "permissions", "if",
        "strategy", "outputs", "steps[<index>]",
    ]]
    + ["others"]
)
entity_table = entity_table.reindex(
    [entity for entity in entity_order if entity in entity_table.index]
)
entity_table.index.name = "row"

entity_table_path = OUT_DIR / "rq3_entity_change_table.csv"
entity_table.to_csv(entity_table_path)

step_rows = entity_table.index.str.startswith("jobs.<job_id>.steps[<index>]")
print(f"Total syntactic entity changes: {total_changes}")
print(f"APR occurrences: {group_totals['ai']}")
print(f"HPR occurrences: {group_totals['human']}")
print(
    "Step-related share: "
    f"APR {entity_table.loc[step_rows, 'APR Total'].sum():.1f}%, "
    f"HPR {entity_table.loc[step_rows, 'HPR Total'].sum():.1f}%"
)
print(f"Table written to {entity_table_path.relative_to(ROOT)}")
display(entity_table)


Total syntactic entity changes: 1692
APR occurrences: 941
HPR occurrences: 751
Step-related share: APR 69.3%, HPR 66.4%
Table written to datasets/processed/rq3/rq3_entity_change_table.csv


,APR Added,APR Changed,APR Removed,APR Total,HPR Added,HPR Changed,HPR Removed,HPR Total,All PRs Total (pooled)
row,,,,,,,,,
name,0.0,1.3,0.1,1.4,0.0,0.4,0.0,0.4,0.9
env,0.4,0.1,0.1,0.6,1.5,0.3,0.4,2.1,1.3
concurrency,0.1,0.0,0.0,0.1,0.0,0.1,0.0,0.1,0.1
on,2.0,1.8,1.3,5.1,3.3,1.5,0.9,5.7,5.4
permissions,0.2,0.0,0.0,0.2,0.5,0.0,0.0,0.5,0.4
jobs.<job_id>,4.1,0.0,1.1,5.2,2.0,0.0,1.2,3.2,4.3
jobs.<job_id>.name,0.0,0.3,0.3,0.6,0.0,0.5,0.0,0.5,0.6
jobs.<job_id>.needs,3.0,0.2,0.0,3.2,0.4,1.3,0.4,2.1,2.7
jobs.<job_id>.env,1.6,0.0,0.1,1.7,0.3,0.3,0.0,0.5,1.2
